# 平台背景与原理

后面几节讲的是**怎么装、怎么跑**；
本节先补上另一半：**这个平台为什么这样设计、内部是怎么工作的**。

读完你应当能回答：

- 为什么要在**没有 NPU** 的环境里做通信验证？
- 平台的**四层架构**分别负责什么？
- 「模拟运行器」和「算法分析器」各自怎么工作？
- 那两个报错案例，问题出在哪一层？

## 本节学习大纲

- 背景：大模型时代为什么需要「纯软件验证」
- 总体架构：四层结构
- 模拟运行器（Runner）：把 API 调用变成可执行任务
- 算法分析器（Checker）：在任务图上做逻辑推演
- 工具使用：`hccl-vm` 的子命令
- 两个实战案例：问题是怎么被提前拦住的
- 与后续各节的衔接
- 课堂小练
- 本节总结


## 1. 背景：大模型时代为什么需要「纯软件验证」

### 1.1 物理验证撞上了三堵墙

智算集群的规模上来之后，**在真机上验证 HCCL 变得越来越难**。难点可以归纳成三个：

| 问题 | 具体表现 |
| :--- | :--- |
| **通信成了瓶颈枢纽** | MoE、All-to-All 这类通信密集的训练里，HCCL 的**微小抖动在千卡规模会被指数级放大** |
| **验证成本攀升** | 真机验证（把程序放到真实硬件上跑）依赖昂贵的集群资源、排队时间长，跟不上敏捷迭代的节奏 |
| **问题定位复杂** | 软硬件耦合极深，很难快速区分到底是「**算法逻辑 Bug**」「**内存并发踩踏**」还是「**底层网络抖动**」 |

还有一个更根本的困境——**出错时，看不到设备内部发生了什么**。

举个具体的：某个搬运任务本该只写自己那段内存，却因为算错了位置，多写了几个字节
（这叫“越界”）。在多卡并发、数据要经网卡和交换机在多张卡之间来回传的环境里，
这一点点越界**不会当场报错**，反而很容易被后续的正常读写淹没——日志看上去一切正常，
等最终结果对不上时，早已错过了能追查的时机。

传统做法只能先把出错那一刻的**内存和运行状态整份保存下来**（这个动作叫 Dump，
可以理解为“给现场拍一张快照”），事后再离线重放、一点点复现。一轮排查动辄要花很久，
定位效率很低。

### 1.2 北极星的新思路：把问题拦在上真机之前

北极星工具（HCCL-VM）的思路是**前置排雷**：

> **把原本必须上真机才能发现的大量逻辑问题，拦截在纯软件分析阶段**，从而释放稀缺的真机算力。

换句话说：**能在电脑上跑出来的问题，就不要拿到真机上去撞**。真机只用来验证「确实需要硬件」
的那部分（真实链路、真实性能）。

> 代码仓：`https://gitcode.com/cann/hcomm/tree/master/test/hccl_vm`

## 2. 总体架构：四层结构

整个平台自上而下分成四层。理解这四层，后面两个工具的定位就清楚了。

![图 1](./images/png/polaris_architecture.png)

*图 1 · 北极星工具（HCCL-VM）总体架构：用户层 → 插件管理层 → 代理 / 虚拟运行时 / 通信层 → 共享内存存储层。*

### 2.1 四层分别负责什么

| 层 | 关键组件 | 职责 |
| :--- | :--- | :--- |
| **用户层** | `hccl-vm` CLI（`start` / `run` / `plugin` / `log` / `table` / `model`）、日志子系统 spdlog | 启动环境、管理插件、查看仿真数据库；日志落盘便于事后排查 |
| **插件管理层** | `HcclPluginManager`（单例）→ **Runner 插件**（模拟运行器）/ **Checker 插件**（算法分析器） | 统一管理插件：安装 / 卸载 / 运行 / 列表；功能按插件扩展 |
| **代理层 / 虚拟运行时层 / 通信层** | Level 1/2 API 拦截 · CCU 执行器 · 顺序执行器 · IPC/SHM | 拦截调用、模拟执行任务、负责进程间通信 |
| **共享内存存储层** | `TaskCollection` / `RunnerDB` / `ProxyBuffer` / `HcclVmFlag` | 存任务元数据、算子元数据、代理缓冲区、状态标志 |

### 2.2 一条数据是怎么流过去的

把四层串起来看，一次验证是这样走的：

1. **用户层** 启动环境，把 **Runner 插件** 装进进程；
2. 测试程序调用 HCCL 算子时，**代理层** 把调用拦下来，提取出「算子信息」和「底层任务」；
3. 这些记录写进 **共享内存存储层**（`RunnerDB` 存算子、`TaskCollection` 存任务）；
4. **虚拟运行时层** 把任务模拟执行一遍，让测试程序能继续跑完并核对结果；
5. 需要更深的检查时，**Checker 插件** 读取同一份记录，在任务图上做推演。

> 日常使用时不必记住这些内部名字，只要抓住一句话：
> **「调用被拦下来 → 变成记录 → 交给执行或分析」**。

## 3. 模拟运行器（Runner）：把 API 调用变成可执行任务

### 3.1 核心机制

> 通过 **`LD_PRELOAD`** 动态库拦截技术，透明捕获 HCCL API 调用，提取任务元数据，
> 在虚拟运行时中模拟执行并验证正确性。

`LD_PRELOAD` 是 Linux 的一个机制：让指定的动态库**先于**系统库被加载，从而"顶替"掉原来的函数。
HCCL-VM 就是用它把 `HcclAllReduce` 这类调用**悄悄接管**——测试程序完全不用改代码。

![图 2](./images/png/polaris_runner.png)

*图 2 · 模拟运行器的执行流程：① 用户程序发起调用 → ② LD_PRELOAD 双层拦截 → ③ 任务收集与分发 → ④ 虚拟运行时执行 → ⑤ 输出执行日志。*

### 3.2 为什么要拦两层

| 拦截层 | 拦什么 | 记录什么 |
| :--- | :--- | :--- |
| **Level 2 · 算子级** | 集合通信算子的整体调用 | 算子的类型、数据描述等——「这次原本要做什么」 |
| **Level 1 · 任务级** | 算子展开后的底层动作 | 搬运、归约、等待、通知等——「准备怎么完成它」 |

两层合起来，既知道**意图**，也知道**具体步骤**。这正是后面能做算法分析的前提。

### 3.3 关键技术点

| 技术 | 实现方式 | 好处 |
| :--- | :--- | :--- |
| 动态库拦截 | `LD_PRELOAD` + `libhccl_proxy_level1/2.so` | **无侵入、透明拦截**，业务代码零改动 |
| 共享内存 | `Boost.Interprocess` | 高效的进程间通信 |
| 柔性数组 | `TaskCollection::tasks[]` | 支持**可变数量**的任务存储 |
| 工厂模式 | `CcuExecutorFactory` | 易于扩展新的指令类型 |

### 3.4 两种运行模式

| 模式 | 环境变量 | 行为 |
| :--- | :--- | :--- |
| **Runner** | `HCCL_VM_MODE=RUNNER` | 指令级模拟执行 |
| **Hybrid** | `HCCL_VM_MODE=HYBRID` | 模拟执行 **+ 校验** |

> 本章后续几节采用的是**插件方式**（`hccl-vm plugin install @runner`）来启用这些能力，
> 不需要手动设置上面的模式变量。了解这两个模式，是为了理解平台内部的功能组合。

## 4. 算法分析器（Checker）：在任务图上做逻辑推演

### 4.1 核心洞察：不跑数值，只推逻辑

Checker 的设计可以概括成两句话：

- **动态时态捕获**：摒弃静态源码推演，通过底层**静默劫持**获取**真实运行时**的 API 调用序列；
- **轻量化推演引擎**：**不需要真实浮点运算与物理周期仿真**，只通过数据结构描述不同任务，
  以极低开销完成逻辑推演。

一句话：**它检查的是「数据怎么流动」，而不是「数值算得对不对」。**

![图 3](./images/png/polaris_analyzer.png)

*图 3 · 算法分析器的四步流水线：输入捕获 → 底座构建 → 前置排雷 → 双路高阶校验。*

### 4.2 四步流水线

| 步骤 | 做什么 | 回答什么问题 |
| :--- | :--- | :--- |
| **① 输入捕获** | 劫持真实 HCCL API，拿到真实调用任务，重组为多组 Task Queues | 实际发生了什么调用？ |
| **② 底座构建** | 提取通信与同步原语，把「通知 / 等待」（Post/Wait）配对起来、建立跨队列的依赖，生成**全局任务图** | 这些调用之间是什么依赖关系？ |
| **③ 前置排雷** | 基础结构校验：排查环形依赖、悬挂配对关系 | **图能不能走通？** |
| **④ 双路高阶校验** | 支线 A（语义）：结果图案是否符合协议；支线 B（内存）：执行中资源是否被踩坏 | 结果对不对 / 过程有没有踩坏？ |

> **为什么必须「先排雷、后校验」？** 因为图本身走不通时，后面算出来的语义 / 内存结论都没有意义。

### 4.3 语义校验：给数据「染色」

语义校验的做法很巧妙：**给每个 rank 的每段输入贴一个来源标签**（这套做法叫「血缘染色」），
再顺着任务看这些标签怎么流动、最后分布成什么样。

![图 4](./images/png/polaris_semantics.png)

*图 4 · 语义校验：A / B 表示两个 rank 的输入来源，数字表示同一输入中的不同位置（AllReduce 以求和为例）。*

| 算子 | 校验时关心什么 |
| :--- | :--- |
| **AllGather** | 每个 rank 都拿到按约定顺序拼接的各 rank 数据——**严格拼接**，来源与位置都不能错 |
| **AllReduce** | 每个 rank 都拿到对应位置上各 rank 输入的归约结果——**全量混合**，贡献必须完整 |
| **ReduceScatter** | 先满足归约语义，再按约定把不同结果块分配给对应 rank |

常见的错误都能被这套方法抓出来：**拷贝位置错了、少了某个 rank 的贡献、某段数据被重复使用**，
最终图案都会与协议不符。

> 注意分工：**Checker 管「来源分布对不对」，数值本身则由 Runner 的模拟执行 + 测试程序核对。**

### 4.4 内存冲突校验：先看会不会并发，再看地址是否重叠

内存冲突校验是语义校验的**补充**，专门盯「执行过程中有没有把内存踩坏」。

判别准则可以概括成两句话：**先看两个任务会不会同时跑，再看它们访问的地址有没有重叠**。

![图 5](./images/png/polaris_memory.png)

*图 5 · 内存冲突校验：左边缺少先后依赖、区间又重叠 → 冲突；右边用通知 / 等待建立了依赖 → 可以安全复用同一区间。*

一个完整的判定需要**三个条件同时成立**：

1. 两个任务**可能并发**（任务图上没有先后路径）；
2. 它们访问了**重叠的内存区间**；
3. 至少有一个是**写**操作。

| 情况 | 判定 |
| :--- | :--- |
| 可能并发 + 区间重叠 + 至少一个写 | **报冲突** |
| 可能并发 + 区间重叠 + 两者都只读 | 不构成读写冲突 |
| 区间重叠，但有依赖保证严格先后 | 不算冲突——**地址重叠本身不是问题，缺少依赖才是** |
| rank 或缓冲区类型不同 | 先区分地址空间，再判断是否真的访问同一段内存 |

> 定位失败时，要同时看：**任务编号 / rank / stream / 缓冲区类型 / 偏移与长度**，
> 以及它们之间有没有正确的同步边。修复方向通常是三者之一：**改数据布局、改访问范围、补同步关系**。

## 5. 工具使用：`hccl-vm` 的子命令

平台的所有能力都挂在 `hccl-vm` 这一个命令下，通过子命令区分。

### 5.1 顶层子命令

| 子命令 | 职责 |
| :--- | :--- |
| `start` | 启动仿真环境并进入交互 shell |
| `run` | 运行任务 |
| `plugin` | 插件安装 / 卸载 / 运行 / 列表 |
| `log` | 查询 / 设置日志等级 |
| `table` | 展示 / 更新仿真数据库表 |
| `model` | 列出建模文件 |
| `mock-comm` | 为**单个算子**建立通信域 |
| `reset` | 清空数据库表、重置仿真环境 |

### 5.2 `plugin` 的三级操作

| 操作 | 作用 |
| :--- | :--- |
| `plugin install @checker` / `@runner` | 安装 checker / runner 插件 |
| `plugin uninstall @checker` / `@runner` | 卸载 checker / runner 插件 |
| `plugin run @checker` / `@runner` | 执行 checker / runner 插件 |

> 详细的安装部署与使用指导见代码仓：
> `https://gitcode.com/cann/hcomm/tree/master/test/hccl_vm`
>
> 装好环境、启动平台后（见后面几节），随时可以用 `hccl-vm --help` 和
> `hccl-vm plugin --help` 查看**当前版本**支持的命令与选项。

## 6. 两个实战案例：问题是怎么被提前拦住的

后面动手运行时可能遇到下面两类报错——一个由 Runner 报出、一个由 Checker 报出，正好体现两者的分工。

### 6.1 模拟运行器报错：CCU 资源不足

运行测试时，模拟运行器可能报出资源相关的错误（关键词）：

```
resource is not enough
resource unavailable, try to fallback
```

**怎么定位**：

1. 先确认**资源类型、需求量、可用量**分别是什么；
2. 再检查**展开模式、模型规格、用例规模**——是不是这次用例要的资源超过了当前配置；
3. **往后看**：回退（fallback）有没有成功？测试最终结果是什么？

> ⚠️ 单条「资源不足」日志**不等于整个用例失败**，也不能直接推断「机器没有物理 NPU」——
> 这是模拟环境的资源配置问题，要看上下文。

### 6.2 算法分析器报错：内存冲突校验不通过

执行 Checker 时，可能报出内存冲突（关键词）：

```
Slice conflict
CheckGraphV3TaskMem failed
```

**怎么定位**：

1. 把**两次访问**的 rank、队列、内存类型、偏移和长度对应起来；
2. 检查它们之间**是否存在正确的依赖**（对照 §4.4 的三个条件）；
3. 修复方向：**改数据布局 / 改访问范围 / 补同步关系**；
4. 改完**重新跑原用例 + Checker** 验证。

> 这两个案例正好演示了平台的两种价值：**Runner 提前暴露资源类问题，Checker 提前暴露算法逻辑问题。**

## 7. 与后续各节的衔接

本节讲**原理**，动手操作放在后面的章节：

| 这里讲的概念 | 后续各节中的位置 |
| :--- | :--- |
| 平台所需的软件环境 | [3.3 获取并解压绿色包](./03.03_cloud_pack_setup.ipynb)（自建包见 [3.9](./03.09_appendix_local_env.ipynb)） |
| 发起通信的用户程序（`hccl_test`） | [3.4 算例说明](./03.04_test_cases.ipynb) |
| 模型、拓扑与通信参与者 | [3.5 启动虚拟集群并选择 rank](./03.05_first_comm_test.ipynb) |
| **模拟运行器（Runner）** | [3.5 运行算例与 Checker](./03.05_first_comm_test.ipynb) |
| **算法分析器（Checker）** | [3.5 执行 Checker 并检查汇总](./03.05_first_comm_test.ipynb) |
| 改完源码重新验证 | [3.9 本地 WSL 环境与自建绿色包（选看）](./03.09_appendix_local_env.ipynb) |

> 阅读顺序建议：**先通读本节了解整体原理**，再按后面几节动手；等环境跑通后回看本节，
> 架构图和两个工具的原理会立刻对得上你敲过的命令。

## 8. 课堂小练

请根据本节学习内容完成以下题目进行自测。

**1、（判断题）** HCCL-VM 必须在真实的 NPU 集群上运行，才能验证通信是否正确。

---

**2、（单选题）** HCCL-VM 能「悄悄接管」测试程序里的 HCCL 调用，靠的是什么机制？

- **A)** 修改测试程序的源码
- **B)** `LD_PRELOAD` 动态库拦截
- **C)** 修改 CANN 库本身
- **D)** 用代理服务器转发调用

---

**3、（单选题）** 关于模拟运行器（Runner）与算法分析器（Checker）的分工，正确的是？

- **A)** 两者都做数值计算
- **B)** Runner 模拟执行任务、让测试程序核对数值；Checker 在任务图上检查语义与内存访问
- **C)** Runner 查内存、Checker 算数值
- **D)** 两者都只负责收集日志

---

**4、（判断题）** Checker 需要真实执行浮点运算，才能判断结果对不对。

---

**5、（简答题）** 内存冲突校验要同时满足哪三个条件，才判定为「冲突」？

**执行以下代码获取答案。**

In [ ]:
# 读取本节课堂小练答案（用 Python 读文件：跨平台，并保证中文不乱码）
import pathlib
_p = pathlib.Path('./answer/03.02_answer.txt')
if _p.exists():
    print(_p.read_text(encoding='utf-8'))
else:
    print(f'未找到答案文件：{_p}')
    print('（请确认 answer/ 目录与本 notebook 在同一层）')

## 9. 本节总结

- **为什么要做纯软件验证**：真机验证难在三点 —— 通信问题在千卡规模会被放大、集群资源贵且要排队、
  软硬件耦合难定位；而微小的越界在真机上很容易被淹没，传统 Dump 重放又慢。
  HCCL-VM 的思路是把问题**拦在上真机之前**。
- **四层架构**：用户层 → 插件管理层 → 代理 / 虚拟运行时 / 通信层 → 共享内存存储层。
  一次验证就是「**调用被拦下来 → 变成记录 → 交给执行或分析**」。
- **模拟运行器（Runner）**：用 `LD_PRELOAD` 透明拦截 HCCL 调用，把算子与底层任务记录下来，
  在虚拟运行时里模拟执行，让测试程序能核对数值。
- **算法分析器（Checker）**：不跑数值，只在**任务图**上做逻辑推演 —— 先排雷（环形依赖、悬挂配对），
  再做语义校验（数据来源分布）与内存冲突校验（并发 + 区间重叠 + 至少一个写）。
- **两者分工**：Runner 管「数值对不对」，Checker 管「数据怎么流动、有没有踩坏内存」。

---

[← 3.1 本章介绍](./03.01_chapter_intro.ipynb) · [返回本章目录](./03.01_chapter_intro.ipynb) · [3.3 获取并解压绿色包 →](./03.03_cloud_pack_setup.ipynb)